In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

# Model 18 - with DEV

In [ ]:
# 1. חיבור לדרייב (תתבקשי לאשר גישה)
from google.colab import drive
drive.mount('/content/drive')

# 2. שדרוג pip והתקנת הספריות
!pip install --upgrade pip
!pip install -q -U bitsandbytes accelerate peft transformers datasets scipy scikit-learn

print("✅ Installation Done. Now please RESTART SESSION (Runtime -> Restart Session).")

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import torch
import os

# ==========================================
# 1. הגדרות - המודל המנצח (V17 / V14-Fixed)
# ==========================================
output_dir = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/Final_Full_Train_Dev"
model_id = "NousResearch/Meta-Llama-3-8B"

# נתיבים
train_path = "/content/drive/MyDrive/Data mining/Text_mining/train.json"
dev_path = "/content/drive/MyDrive/Data mining/Text_mining/dev.json"

print(f"🚀 Starting FINAL training (Train + Dev) at: {output_dir}")

# ==========================================
# 2. איחוד הדאטה (Train + Dev) 🤝
# ==========================================
def load_and_fix_data(path):
    try:
        df = pd.read_json(path)
    except ValueError:
        df = pd.read_json(path, orient='index')
    if df.shape[1] > 100 and df.shape[0] < df.shape[1]:
        df = df.T
    if 'label' not in df.columns:
        if 'average' in df.columns: df['label'] = df['average']
        elif 'score' in df.columns: df['label'] = df['score']
    return df

print("📊 Loading and Merging Data...")
df_train = load_and_fix_data(train_path)
df_dev = load_and_fix_data(dev_path)

# האיחוד הגדול:
df_full = pd.concat([df_train, df_dev], ignore_index=True)
print(f"✅ Merged Data Size: {len(df_full)} examples (Train: {len(df_train)} + Dev: {len(df_dev)})")

def format_data(row):
    score = row.get('label')
    ending = row.get('ending')
    if ending is None: ending = ""
    precontext = row.get('precontext', '')
    sentence = row.get('sentence', '')
    homonym = row.get('homonym', '')
    definition = row.get('judged_meaning', '')

    has_ending = str(ending).strip() != ""
    if has_ending:
        text = f"### Instruction:\nAnalyze consistency.\nContext: {precontext} {sentence}\nTarget: {homonym} ({definition})\nEnding: {ending}"
    else:
        text = f"### Instruction:\nEvaluate fit.\nContext: {precontext}\nTarget: {homonym} ({definition})\nSentence: {sentence}"
    return {"text": text, "label": float(score)}

# יצירת דאטה-סט אחד גדול
full_ds = Dataset.from_list([format_data(row) for _, row in df_full.iterrows()])

# ==========================================
# 3. המודל (בדיוק כמו V17 המנצח)
# ==========================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
    problem_type="regression",
    torch_dtype=torch.float16
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    modules_to_save=["score"] # חובה!
)
model = get_peft_model(model, peft_config)

# ==========================================
# 4. אימון "עיוור" (בלי ולידציה)
# ==========================================
tokenizer_func = lambda x: tokenizer(x["text"], padding="max_length", truncation=True, max_length=512)
tokenized_full = full_ds.map(tokenizer_func, batched=True)

args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-4, # לא לשנות! זה מה שעבד.
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3, # משאירים אותו מספר אפוקים (סה"כ צעדים יגדל קצת וזה טוב)
    optim="paged_adamw_8bit",

    # שינויים לאימון מלא:
    eval_strategy="no", # אין ולידציה
    save_strategy="steps",
    save_steps=100,     # נשמור כל 100 צעדים ליתר ביטחון
    save_total_limit=2,

    fp16=False,
    gradient_checkpointing=True,
    logging_steps=20,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_full,
    # אין eval_dataset ואין compute_metrics
)

print("🚀 Starting FINAL Full Training...")
trainer.train()

print(f"💾 Saving final model to {output_dir}")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("🏆 Done! Ready for Test Inference.")

stopped at step 920 when GPU ended, new code to cintinue from where we stopped

In [ ]:
import os
import sys
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ==========================================
# 1. הגדרות שחזור (Recovery)
# ==========================================
# שימי לב: זה הנתיב של האימון המלא
output_dir = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/Final_Full_Train_Dev"
model_id = "NousResearch/Meta-Llama-3-8B"
train_path = "/content/drive/MyDrive/Data mining/Text_mining/train.json"
dev_path = "/content/drive/MyDrive/Data mining/Text_mining/dev.json"

print(f"🚑 RECOVERY MODE: Checking for checkpoints in {output_dir}")

# ==========================================
# 2. טעינת דאטה מחדש (חובה בסשן חדש)
# ==========================================
def load_and_fix_data(path):
    try:
        df = pd.read_json(path)
    except ValueError:
        df = pd.read_json(path, orient='index')
    if df.shape[1] > 100 and df.shape[0] < df.shape[1]:
        df = df.T
    if 'label' not in df.columns:
        if 'average' in df.columns: df['label'] = df['average']
        elif 'score' in df.columns: df['label'] = df['score']
    return df

print("📊 Reloading Train + Dev Data...")
df_train = load_and_fix_data(train_path)
df_dev = load_and_fix_data(dev_path)
df_full = pd.concat([df_train, df_dev], ignore_index=True)

def format_data(row):
    score = row.get('label')
    ending = row.get('ending')
    if ending is None: ending = ""
    precontext = row.get('precontext', '')
    sentence = row.get('sentence', '')
    homonym = row.get('homonym', '')
    definition = row.get('judged_meaning', '')
    has_ending = str(ending).strip() != ""
    if has_ending:
        text = f"### Instruction:\nAnalyze consistency.\nContext: {precontext} {sentence}\nTarget: {homonym} ({definition})\nEnding: {ending}"
    else:
        text = f"### Instruction:\nEvaluate fit.\nContext: {precontext}\nTarget: {homonym} ({definition})\nSentence: {sentence}"
    return {"text": text, "label": float(score)}

full_ds = Dataset.from_list([format_data(row) for _, row in df_full.iterrows()])

# ==========================================
# 3. הגדרת המודל מחדש
# ==========================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
    problem_type="regression",
    torch_dtype=torch.float16
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    modules_to_save=["score"]
)
model = get_peft_model(model, peft_config)

# ==========================================
# 4. הגדרת המאמן (Trainer)
# ==========================================
tokenizer_func = lambda x: tokenizer(x["text"], padding="max_length", truncation=True, max_length=512)
tokenized_full = full_ds.map(tokenizer_func, batched=True)

args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    optim="paged_adamw_8bit",
    eval_strategy="no",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=False,
    gradient_checkpointing=True,
    logging_steps=20,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_full,
)

# ==========================================
# 5. זיהוי הצ'קפוינט והמשך ריצה (Magic Fix) 🪄
# ==========================================
print(f"🕵️‍♀️ Scanning for checkpoints in: {output_dir}")
last_checkpoint = None

if os.path.isdir(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint")]
    if len(checkpoints) > 0:
        # מיון כדי למצוא את האחרון
        checkpoints.sort(key=lambda x: int(x.split('-')[1]))
        last_checkpoint = os.path.join(output_dir, checkpoints[-1])
        print(f"✅ Found latest checkpoint: {checkpoints[-1]}")
        print("🚀 RESUMING TRAINING from there...")
    else:
        print("⚠️ No checkpoints found! Starting fresh (only if folder is empty).")
else:
    print("⚠️ Directory not found.")

# הפקודה הקריטית
trainer.train(resume_from_checkpoint=last_checkpoint)

print(f"💾 Saving final model to {output_dir}")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("🏆 Done!")

In [ ]:
import pandas as pd
import json
import os
import torch
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ==========================================
# 1. הגדרות נתיבים (V18 - המודל הסופי)
# ==========================================
# הנתיב שבו נשמר המודל של TRAIN + DEV
model_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/Final_Full_Train_Dev"

# נתיב קובץ הטסט
test_path = "/content/drive/MyDrive/Data mining/Text_mining/test.json"

# נתיב שמירת התוצאות
output_dir = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V18_Final_Submission"
os.makedirs(output_dir, exist_ok=True)

print(f"🚀 Starting Inference with V18 (Full Train+Dev Model)")
print(f"📂 Loading model from: {model_path}")

# ==========================================
# 2. טעינת המודל
# ==========================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_path)
base_model = AutoModelForSequenceClassification.from_pretrained(
    "NousResearch/Meta-Llama-3-8B",
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
    problem_type="regression",
    torch_dtype=torch.float16
)
base_model.config.pad_token_id = tokenizer.pad_token_id

# טעינת המוח (LoRA) והפה (Score Head) של V18
model = PeftModel.from_pretrained(base_model, model_path)
model.eval()

print("✅ V18 Model loaded successfully!")

# ==========================================
# 3. טעינת קובץ ה-Test
# ==========================================
print("📊 Loading Test Data...")
try:
    df_test = pd.read_json(test_path, orient='index')
except ValueError:
    print("⚠️ Format warning, trying default load...")
    df_test = pd.read_json(test_path)

print(f"✅ Loaded {len(df_test)} test examples.")

# ==========================================
# 4. פונקציית החיזוי
# ==========================================
def predict_consistency(context, sentence, homonym, definition, ending=""):
    # טיפול בערכים חסרים ב-Test set
    # אם הסוף הוא סימני שאלה או ריק, נשתמש בפרומפט הקצר
    if str(ending).strip() == "(???)" or str(ending).strip() == "":
        text = f"### Instruction:\nEvaluate fit.\nContext: {context}\nTarget: {homonym} ({definition})\nSentence: {sentence}"
    else:
        text = f"### Instruction:\nAnalyze consistency.\nContext: {context} {sentence}\nTarget: {homonym} ({definition})\nEnding: {ending}"

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to("cuda")

    with torch.no_grad():
        outputs = model(**inputs)

    score = outputs.logits.item()
    return max(1.0, min(5.0, score)) # מגביל לטווח 1-5

# ==========================================
# 5. ריצה ושמירה
# ==========================================
submission_data = []
full_csv_data = []

print("🚀 Running predictions...")

for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    # שליפת נתונים
    context = row.get('precontext', '')
    sentence = row.get('sentence', '')
    homonym = row.get('homonym', '')
    definition = row.get('judged_meaning', '')
    ending = row.get('ending', '')

    # חיזוי
    pred = predict_consistency(context, sentence, homonym, definition, ending)

    # זיהוי ה-ID (חשוב מאוד להגשה!)
    # נשתמש באינדקס ("0", "1") כי זה המפתח הראשי ב-JSON
    row_id = str(index)

    # שמירה ל-JSONL (להגשה)
    submission_data.append({
        "id": row_id,
        "prediction": pred
    })

    # שמירה ל-CSV (לבדיקה עצמית)
    row_dict = row.to_dict()
    row_dict['prediction'] = pred
    row_dict['id_key'] = row_id
    full_csv_data.append(row_dict)

# ==========================================
# 6. יצירת הקבצים
# ==========================================
# קובץ JSONL להגשה
jsonl_path = os.path.join(output_dir, "predictions.jsonl")
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for entry in submission_data:
        f.write(json.dumps(entry) + '\n')

# קובץ CSV לגיבוי
csv_path = os.path.join(output_dir, "v18_predictions_full.csv")
pd.DataFrame(full_csv_data).to_csv(csv_path, index=False)

print("\n" + "="*30)
print(f"🏆 DONE! Inference finished.")
print(f"📄 Submission File: {jsonl_path}")
print(f"📊 Debug File: {csv_path}")
print("="*30)
print("👉 Reminder: ZIP only the 'predictions.jsonl' file (not the folder) and upload!")

# Enssamble ModelA and ModelB

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import os
import json

# ==========================================
# 1. הגדרת נתיבים (נא לעדכן את הנתיבים החסרים!)
# ==========================================

# --- נתוני אמת (Ground Truth) ---
dev_ground_truth_path = "/content/drive/MyDrive/Data mining/Text_mining/dev.json"

# --- פרדיקציות על ה-DEV (לצורך מציאת המשקל) ---
# מודל A (החזק, 0.79)
model_a_dev_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V17/predictions.jsonl"
# מודל B (הלוקאלי, 0.77) - שימי כאן את הנתיב לקובץ שהעלית
model_b_dev_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelB/dev_predictions.jsonl"

# --- פרדיקציות על ה-TEST (שאותן נגיש) ---
# מודל A (V18 - Train+Dev)
model_a_test_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V18_Final_Submission/predictions.jsonl"
# מודל B (Train+Dev) - שימי כאן את הנתיב לקובץ שהעלית
model_b_test_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelB/submission.jsonl"

output_dir = "/content/drive/MyDrive/Data mining/Text_mining/Ensemble_Submission"
os.makedirs(output_dir, exist_ok=True)

# ==========================================
# 2. פונקציית טעינה מתוקנת (כולל Transpose)
# ==========================================
def load_data(path, is_ground_truth=False):
    print(f"📂 Loading: {path}")

    # אם זה קובץ JSONL (הפרדיקציות)
    if path.endswith('.jsonl'):
        data = []
        with open(path, 'r') as f:
            for line in f:
                data.append(json.loads(line))
        df = pd.DataFrame(data)

    # אם זה קובץ JSON רגיל (dev.json)
    else:
        try:
            df = pd.read_json(path)
        except ValueError:
            df = pd.read_json(path, orient='index')

        # --- התיקון הקריטי: הפיכת הטבלה אם היא "שוכבת" ---
        # אם יש יותר עמודות משורות, כנראה שצריך להפוך
        if df.shape[1] > df.shape[0]:
            print("   🔄 Transposing data frame...")
            df = df.T

        # חיפוש עמודת הציון
        if is_ground_truth:
            found = False
            for name in ['label', 'score', 'average', 'mos']:
                if name in df.columns:
                    df['label'] = df[name]
                    found = True
                    print(f"   ✅ Found label column: '{name}'")
                    break
            if not found:
                print(f"   ❌ ERROR: Label column not found in {df.columns.tolist()}")

    # המרת ID לטקסט
    if 'id' in df.columns:
        df['id'] = df['id'].astype(str)
    else:
        df['id'] = df.index.astype(str)

    return df

# ==========================================
# 3. ביצוע האנסמבל
# ==========================================

# 1. טעינת הנתונים
print("\n📊 --- Loading Data ---")
df_truth = load_data(dev_ground_truth_path, is_ground_truth=True)
df_a_dev = load_data(model_a_dev_path)
df_b_dev = load_data(model_b_dev_path)

# 2. מיזוג
merged_dev = df_truth[['id', 'label']].merge(df_a_dev[['id', 'prediction']], on='id', suffixes=('', '_A'))
merged_dev = merged_dev.merge(df_b_dev[['id', 'prediction']], on='id', suffixes=('_A', '_B'))

# סידור שמות
if 'prediction' in merged_dev.columns:
     merged_dev.rename(columns={'prediction': 'pred_b'}, inplace=True)
else:
     merged_dev.rename(columns={'prediction_A': 'pred_a', 'prediction_B': 'pred_b'}, inplace=True)

print(f"✅ Merged {len(merged_dev)} rows.")

# 3. חיפוש משקל
best_score = -1
best_weight = 0.5

print("\n🔎 --- Searching for Optimal Weight ---")
for w in np.arange(0.0, 1.05, 0.05):
    blended_preds = (merged_dev['pred_a'] * w) + (merged_dev['pred_b'] * (1 - w))
    current_score, _ = spearmanr(merged_dev['label'], blended_preds)
    if current_score > best_score:
        best_score = current_score
        best_weight = w

print("-" * 30)
print(f"🏆 Best Result Found!")
print(f"   Weight for Model A: {best_weight:.2f}")
print(f"   Weight for Model B: {1 - best_weight:.2f}")
print(f"   New Spearman:       {best_score:.4f}")
print("-" * 30)

# 4. יצירת ההגשה (TEST)
print("\n🚀 --- Generating Submission ---")
df_a_test = load_data(model_a_test_path)
df_b_test = load_data(model_b_test_path)

merged_test = df_a_test.merge(df_b_test, on='id', suffixes=('_A', '_B'))

merged_test['final_prediction'] = (merged_test['prediction_A'] * best_weight) + \
                                  (merged_test['prediction_B'] * (1 - best_weight))

submission_data = []
for _, row in merged_test.iterrows():
    submission_data.append({
        "id": row['id'],
        "prediction": float(row['final_prediction'])
    })

output_path = os.path.join(output_dir, "ensemble_predictions.jsonl")
with open(output_path, 'w', encoding='utf-8') as f:
    for entry in submission_data:
        f.write(json.dumps(entry) + '\n')

print(f"\n💾 Saved to: {output_path}")
print("Done! 🎉")

In [ ]:
import pandas as pd
import numpy as np
import json

# ==========================================
# 1. הגדרת נתיבים (ודאי שהם נכונים אצלך)
# ==========================================
dev_truth_path = "/content/drive/MyDrive/Data mining/Text_mining/dev.json"
model_a_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V17/predictions.jsonl"
model_b_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelB/dev_predictions.jsonl"

# ==========================================
# 2. פונקציית טעינה (אותה אחת שעבדה לנו)
# ==========================================
def load_data(path, is_ground_truth=False):
    if path.endswith('.jsonl'):
        data = []
        with open(path, 'r') as f:
            for line in f:
                data.append(json.loads(line))
        df = pd.DataFrame(data)
    else:
        try:
            df = pd.read_json(path)
        except ValueError:
            df = pd.read_json(path, orient='index')

        if df.shape[1] > df.shape[0]:
            df = df.T

        if is_ground_truth:
            # זיהוי עמודת הציון
            for name in ['label', 'score', 'average', 'mos']:
                if name in df.columns:
                    df['label'] = df[name]
                    break

    if 'id' in df.columns:
        df['id'] = df['id'].astype(str)
    else:
        df['id'] = df.index.astype(str)
    return df

# ==========================================
# 3. חישוב הדיוק
# ==========================================
print("📊 Loading Data...")
df_truth = load_data(dev_truth_path, is_ground_truth=True)
df_a = load_data(model_a_path)
df_b = load_data(model_b_path)

# מיזוג
merged = df_truth[['id', 'label']].merge(df_a[['id', 'prediction']], on='id', suffixes=('', '_A'))
merged = merged.merge(df_b[['id', 'prediction']], on='id', suffixes=('_A', '_B'))

# תיקון שמות
if 'prediction' in merged.columns: merged.rename(columns={'prediction': 'pred_b'}, inplace=True)
else: merged.rename(columns={'prediction_A': 'pred_a', 'prediction_B': 'pred_b'}, inplace=True)

# שימוש במשקולות שמצאנו (0.45 ל-A ו-0.55 ל-B)
w = 0.45
merged['ensemble_pred'] = (merged['pred_a'] * w) + (merged['pred_b'] * (1 - w))

print(f"\n✅ Analyzed {len(merged)} examples.")

# --- בדיקה 1: דיוק בינארי (Binary Accuracy) ---
# הנחת עבודה: הציון הוא בין 1 ל-4. החציון הוא 2.5.
# כל מה שמעל 2.5 נחשב "עקבי" (1), כל מה שמתחת נחשב "לא עקבי" (0).
threshold = 2.5

merged['true_bin'] = (merged['label'] > threshold).astype(int)
merged['ens_bin'] = (merged['ensemble_pred'] > threshold).astype(int)
merged['model_a_bin'] = (merged['pred_a'] > threshold).astype(int)

acc_binary = np.mean(merged['true_bin'] == merged['ens_bin'])
acc_model_a = np.mean(merged['true_bin'] == merged['model_a_bin'])

print("\n🎯 --- Binary Accuracy Results (Threshold 2.5) ---")
print(f"Model A Accuracy:  {acc_model_a:.4f}")
print(f"Ensemble Accuracy: {acc_binary:.4f}")

# --- בדיקה 2: דיוק בעיגול (Rounding Accuracy) ---
# מעגלים למספר השלם הקרוב (למשל 3.7 הופך ל-4) ובודקים אם זה זהה למקור
merged['true_round'] = merged['label'].round()
merged['ens_round'] = merged['ensemble_pred'].round()
merged['model_a_round'] = merged['pred_a'].round()

acc_round_ens = np.mean(merged['true_round'] == merged['ens_round'])
acc_round_a = np.mean(merged['true_round'] == merged['model_a_round'])

print("\n🔢 --- Rounding Accuracy Results (Exact Match) ---")
print(f"Model A Accuracy:  {acc_round_a:.4f}")
print(f"Ensemble Accuracy: {acc_round_ens:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import json
import statistics
from scipy.stats import spearmanr

# ==========================================
# 1. הגדרות נתיבים
# ==========================================
dev_truth_path = "/content/drive/MyDrive/Data mining/Text_mining/dev.json"
model_a_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V17/predictions.jsonl"
model_b_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelB/dev_predictions.jsonl"

# ==========================================
# 2. פונקציות העזר המקוריות (מהקוד שלך)
# ==========================================
def get_standard_deviation(l):
    if len(l) < 2: return 0.0
    return statistics.stdev(l)

def get_average(l):
    return sum(l)/len(l)

def is_within_standard_deviation(prediction, labels):
    avg = get_average(labels)
    stdev = get_standard_deviation(labels)
    # טווח סתיית תקן
    if (avg - stdev) < prediction < (avg + stdev):
        return True
    # או מרחק קטן מ-1
    if abs(avg - prediction) < 1:
        return True
    return False

# ==========================================
# 3. טעינה ומיזוג חכם
# ==========================================
def load_and_prep(truth_path, a_path, b_path):
    print("📊 Loading Data...")

    # 1. טעינת Truth (dev.json)
    try:
        df_truth = pd.read_json(truth_path)
    except ValueError:
        df_truth = pd.read_json(truth_path, orient='index')

    # הפיכה אם צריך (אם נטען "על הצד")
    if df_truth.shape[1] > df_truth.shape[0]:
        df_truth = df_truth.T

    # וידוא שיש עמודת ID ועמודת choices
    if 'id' not in df_truth.columns:
        df_truth['id'] = df_truth.index.astype(str)
    else:
        df_truth['id'] = df_truth['id'].astype(str)

    print(f"   ✅ Ground Truth loaded: {len(df_truth)} rows")

    # 2. טעינת מודל A
    with open(a_path, 'r') as f:
        data_a = [json.loads(line) for line in f]
    df_a = pd.DataFrame(data_a)
    df_a['id'] = df_a['id'].astype(str)

    # 3. טעינת מודל B
    with open(b_path, 'r') as f:
        data_b = [json.loads(line) for line in f]
    df_b = pd.DataFrame(data_b)
    df_b['id'] = df_b['id'].astype(str)

    # 4. מיזוג זהיר (Merge on ID)
    # זה השלב שמונע את הבלאגן - אנחנו ממזגים רק לפי ID משותף
    merged = df_truth[['id', 'choices']].merge(df_a[['id', 'prediction']], on='id', suffixes=('', '_A'))
    merged = merged.merge(df_b[['id', 'prediction']], on='id', suffixes=('_A', '_B'))

    # סידור שמות
    if 'prediction' in merged.columns:
        merged.rename(columns={'prediction': 'pred_b'}, inplace=True)
    else:
        merged.rename(columns={'prediction_A': 'pred_a', 'prediction_B': 'pred_b'}, inplace=True)

    return merged

# ==========================================
# 4. החישוב עצמו
# ==========================================
df = load_and_prep(dev_truth_path, model_a_path, model_b_path)

print(f"🔗 Merged successfully: {len(df)} rows matched.")

# חישוב האנסמבל (הנוסחה שלך)
# Model A: 45%, Model B: 55%
df['ensemble_pred'] = (df['pred_a'] * 0.45) + (df['pred_b'] * 0.55)

# חישוב המדדים
correct_count = 0
gold_averages = []
ensemble_preds = []

for _, row in df.iterrows():
    # המרת choices לרשימה אם צריך
    choices = row['choices']
    if isinstance(choices, str): choices = eval(choices) # במקרה שנטען כמחרוזת

    pred = row['ensemble_pred']

    # 1. חישוב Accuracy
    if is_within_standard_deviation(pred, choices):
        correct_count += 1

    # 2. הכנה ל-Spearman
    gold_averages.append(get_average(choices))
    ensemble_preds.append(pred)

# תוצאות סופיות
final_accuracy = correct_count / len(df)
final_spearman, _ = spearmanr(gold_averages, ensemble_preds)

print("\n" + "="*40)
print(f"🎯 CORRECTED ENSEMBLE RESULTS (Weights: A=0.45, B=0.55)")
print("="*40)
print(f"✅ Accuracy (Within SD): {final_accuracy:.4f} ({correct_count}/{len(df)})")
print(f"📈 Spearman Correlation: {final_spearman:.4f}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import json
import statistics
from scipy.stats import spearmanr

# ==========================================
# 1. Paths
# ==========================================
dev_truth_path = "/content/drive/MyDrive/Data mining/Text_mining/dev.json"
model_b_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelB/dev_predictions.jsonl"

# ==========================================
# 2. Helper Functions (Logic from previous turns)
# ==========================================
def get_standard_deviation(l):
    if len(l) < 2: return 0.0
    return statistics.stdev(l)

def get_average(l):
    return sum(l)/len(l)

def is_within_standard_deviation(prediction, labels):
    avg = get_average(labels)
    stdev = get_standard_deviation(labels)
    # Range of average +/- standard deviation
    if (avg - stdev) < prediction < (avg + stdev):
        return True
    # Or absolute distance less than 1
    if abs(avg - prediction) < 1:
        return True
    return False

# ==========================================
# 3. Load and Prepare Data
# ==========================================
def load_and_prep_single_model(truth_path, pred_path):
    print("📊 Loading Data...")

    # Load Truth
    try:
        df_truth = pd.read_json(truth_path)
    except ValueError:
        df_truth = pd.read_json(truth_path, orient='index')

    if df_truth.shape[1] > df_truth.shape[0]:
        df_truth = df_truth.T

    if 'id' not in df_truth.columns:
        df_truth['id'] = df_truth.index.astype(str)
    else:
        df_truth['id'] = df_truth['id'].astype(str)

    print(f"   ✅ Ground Truth loaded: {len(df_truth)} rows")

    # Load Model Predictions
    with open(pred_path, 'r') as f:
        data_pred = [json.loads(line) for line in f]
    df_pred = pd.DataFrame(data_pred)
    df_pred['id'] = df_pred['id'].astype(str)

    # Merge
    merged = df_truth[['id', 'choices']].merge(df_pred[['id', 'prediction']], on='id')

    return merged

# ==========================================
# 4. Calculate Metrics for Model B
# ==========================================
try:
    df = load_and_prep_single_model(dev_truth_path, model_b_path)
    print(f"🔗 Merged successfully: {len(df)} rows matched.")

    correct_count = 0
    gold_averages = []
    predictions = []

    for _, row in df.iterrows():
        choices = row['choices']
        if isinstance(choices, str): choices = eval(choices)

        pred = float(row['prediction'])

        # Accuracy
        if is_within_standard_deviation(pred, choices):
            correct_count += 1

        # Spearman prep
        gold_averages.append(get_average(choices))
        predictions.append(pred)

    final_accuracy = correct_count / len(df)
    final_spearman, _ = spearmanr(gold_averages, predictions)

    print("\n" + "="*40)
    print(f"📊 MODEL B RESULTS (Dev Set)")
    print("="*40)
    print(f"✅ Accuracy (Within SD): {final_accuracy:.4f} ({correct_count}/{len(df)})")
    print(f"📈 Spearman Correlation: {final_spearman:.4f}")
    print("="*40)

except Exception as e:
    print(f"Error: {e}")

# Enssamble ModelA & ModelB version 2

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import statistics
from scipy.stats import spearmanr

# ==========================================
# 1. הגדרות נתיבים (המעודכנים)
# ==========================================
dev_truth_path = "/content/drive/MyDrive/Data mining/Text_mining/dev.json"

# --- נתוני DEV (למציאת המשקל) ---
model_a_dev_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V17/predictions.jsonl"
model_b_dev_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelB/V2/preds_HYBRID_ENSEMBLE_DEV.jsonl"

# --- נתוני TEST (להגשה) ---
model_a_test_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V18_Final_Submission/predictions.jsonl"
model_b_test_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelB/V2/submission_HYBRID_FINAL_TEST.jsonl"

# תיקייה לשמירה
output_dir = "/content/drive/MyDrive/Data mining/Text_mining/Final_Hybrid_Ensemble"
os.makedirs(output_dir, exist_ok=True)

# ==========================================
# 2. פונקציות הערכה רשמיות (Accuracy within SD)
# ==========================================
def get_standard_deviation(l):
    if len(l) < 2: return 0.0
    return statistics.stdev(l)

def get_average(l):
    return sum(l)/len(l)

def is_within_standard_deviation(prediction, labels):
    avg = get_average(labels)
    stdev = get_standard_deviation(labels)
    # טווח סתיית תקן
    if (avg - stdev) < prediction < (avg + stdev): return True
    # מרחק אבסולוטי קטן מ-1
    if abs(avg - prediction) < 1: return True
    return False

# ==========================================
# 3. פונקציית טעינה חכמה
# ==========================================
def load_data(path, is_ground_truth=False):
    print(f"📂 Loading: {path}")
    data = []

    # טעינה
    if path.endswith('.jsonl'):
        with open(path, 'r') as f:
            data = [json.loads(line) for line in f]
        df = pd.DataFrame(data)
    else:
        try:
            df = pd.read_json(path)
        except ValueError:
            df = pd.read_json(path, orient='index')
        if df.shape[1] > df.shape[0]:
            df = df.T

    # זיהוי עמודות ב-Ground Truth
    if is_ground_truth:
        if 'id' not in df.columns: df['id'] = df.index.astype(str)
        # ודוא ש-choices קיים
        if 'choices' not in df.columns:
            print("⚠️ Warning: 'choices' column missing in truth file.")

    # המרת ID לטקסט
    if 'id' in df.columns: df['id'] = df['id'].astype(str)

    return df

# ==========================================
# 4. ביצוע האופטימיזציה (DEV)
# ==========================================
print("\n🔄 --- Step 1: Optimizing Weights on DEV ---")

df_truth = load_data(dev_truth_path, is_ground_truth=True)
df_a = load_data(model_a_dev_path)
df_b = load_data(model_b_dev_path)

# מיזוג שלושת הטבלאות
merged = df_truth[['id', 'choices']].merge(df_a[['id', 'prediction']], on='id', suffixes=('', '_A'))
merged = merged.merge(df_b[['id', 'prediction']], on='id', suffixes=('_A', '_B'))

# תיקון שמות עמודות אם המיזוג הסתבך
if 'prediction' in merged.columns: merged.rename(columns={'prediction': 'pred_b'}, inplace=True)
else: merged.rename(columns={'prediction_A': 'pred_a', 'prediction_B': 'pred_b'}, inplace=True)

print(f"✅ Merged {len(merged)} rows for optimization.")

best_spearman = -1
best_weight_a = 0.5
best_accuracy = 0.0

# לולאת חיפוש
print("🔎 Searching for optimal weight...")
for w in np.arange(0.0, 1.01, 0.01): # בדיקה בקפיצות של 1%
    blended = (merged['pred_a'] * w) + (merged['pred_b'] * (1 - w))

    # חישוב Spearman (מהיר)
    gold_avgs = [get_average(c if isinstance(c, list) else eval(c)) for c in merged['choices']]
    curr_spearman, _ = spearmanr(gold_avgs, blended)

    if curr_spearman > best_spearman:
        best_spearman = curr_spearman
        best_weight_a = w

        # חישוב Accuracy רק למנצח (כבד יותר לחישוב)
        correct = 0
        for i, row in merged.iterrows():
            choices = row['choices']
            if isinstance(choices, str): choices = eval(choices)
            if is_within_standard_deviation(blended[i], choices):
                correct += 1
        best_accuracy = correct / len(merged)

print("-" * 40)
print(f"🏆 BEST WEIGHT FOUND:")
print(f"   Weight for Model A: {best_weight_a:.2f}")
print(f"   Weight for Model B: {1 - best_weight_a:.2f}")
print(f"   DEV Spearman: {best_spearman:.4f}")
print(f"   DEV Accuracy: {best_accuracy:.4f}")
print("-" * 40)

# ==========================================
# 5. יצירת ההגשה הסופית (TEST)
# ==========================================
print("\n🚀 --- Step 2: Generating Final TEST Submission ---")

df_a_test = load_data(model_a_test_path)
df_b_test = load_data(model_b_test_path)

merged_test = df_a_test.merge(df_b_test, on='id', suffixes=('_A', '_B'))

# חישוב סופי עם המשקל שמצאנו
merged_test['final_prediction'] = (merged_test['prediction_A'] * best_weight_a) + \
                                  (merged_test['prediction_B'] * (1 - best_weight_a))

submission_data = []
for _, row in merged_test.iterrows():
    submission_data.append({
        "id": row['id'],
        "prediction": float(row['final_prediction'])
    })

output_path = os.path.join(output_dir, "predictions.jsonl")
with open(output_path, 'w', encoding='utf-8') as f:
    for entry in submission_data:
        f.write(json.dumps(entry) + '\n')

print("\n" + "="*40)
print(f"💾 FILE SAVED: {output_path}")
print("="*40)
print("👉 Download, Zip, Upload. Good Luck!")

# Model 19 without DEV (V17 improvement)

results: {"accuracy": 0.7959183673469388, "spearman": 0.6943146606272219}

In [ ]:
# 1. חיבור לדרייב (תתבקשי לאשר גישה)
from google.colab import drive
drive.mount('/content/drive')

# 2. שדרוג pip והתקנת הספריות
!pip install --upgrade pip
!pip install -q -U bitsandbytes accelerate peft transformers datasets scipy scikit-learn

print("✅ Installation Done. Now please RESTART SESSION (Runtime -> Restart Session).")

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import gc
from datasets import Dataset
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ==========================================
# 1. הגדרות בסיס
# ==========================================
# מומלץ לשנות את שם התיקייה לריצה נקייה
output_dir = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V19"
model_id = "NousResearch/Meta-Llama-3-8B"
train_path = "/content/drive/MyDrive/Data mining/Text_mining/train.json"
dev_path = "/content/drive/MyDrive/Data mining/Text_mining/dev.json"

# ניקוי זיכרון לפני תחילת העבודה
torch.cuda.empty_cache()
gc.collect()

# ==========================================
# 2. הכנת הדאטה (לוגיקת Ending משופרת)
# ==========================================
def load_and_fix_data(path):
    try:
        df = pd.read_json(path)
    except ValueError:
        df = pd.read_json(path, orient='index')
    if df.shape[1] > 100 and df.shape[0] < df.shape[1]:
        df = df.T
    if 'label' not in df.columns:
        df['label'] = df['average'] if 'average' in df.columns else df['score']
    return df

def format_data(row):
    score = row.get('label')
    precontext = str(row.get('precontext', '')).strip()
    sentence = str(row.get('sentence', '')).strip()
    ending = row.get('ending', '')
    homonym = row.get('homonym', '')
    definition = row.get('judged_meaning', '')

    has_ending = pd.notna(ending) and str(ending).strip() != ""

    if has_ending:
        instruction = "Task: Evaluate story consistency.\nScale: 1.0 (Nonsensical) to 5.0 (Logical)."
        context_text = f"Context: {precontext}\nSentence: {sentence}\nEnding: {ending}"
    else:
        instruction = "Task: Evaluate sentence fit.\nScale: 1.0 (No Fit) to 5.0 (Perfect Fit)."
        context_text = f"Context: {precontext}\nTarget Sentence: {sentence}"

    text = (f"### Instruction:\n{instruction}\n\n"
            f"### Target Word: {homonym}\n### Definition: {definition}\n\n"
            f"### Story:\n{context_text}\n\n"
            f"### Consistency Score (1-5):")
    return {"text": text, "label": float(score)}

print("📊 Loading data...")
df_train = load_and_fix_data(train_path)
df_dev = load_and_fix_data(dev_path)

train_ds = Dataset.from_list([format_data(row) for _, row in df_train.iterrows()])
dev_ds = Dataset.from_list([format_data(row) for _, row in df_dev.iterrows()])

# ==========================================
# 3. אתחול מודל ו-LoRA (עם הבטחה שהראש נשמר)
# ==========================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.problem_type = "regression" # קריטי למניעת תוצאות אקראיות

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["score"], # מוודא ששכבת ה-Regression נשמרת ומתאמנת ב-FP32
    bias="none"
)

model = get_peft_model(model, peft_config)

# בדיקת תקינות הראש
for name, param in model.named_parameters():
    if "score" in name:
        param.requires_grad = True
        print(f"✅ Confirmed: {name} is trainable (Regression Head Safe)")

# ==========================================
# 4. זיהוי צ'קפוינט והגדרות אימון
# ==========================================
last_checkpoint = None
if os.path.exists(output_dir):
    checkpoints = [os.path.join(output_dir, d) for d in os.listdir(output_dir) if d.startswith("checkpoint")]
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
        last_checkpoint = checkpoints[-1]
        print(f"🔄 Resuming from checkpoint: {last_checkpoint}")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.clip(predictions.flatten(), 1.0, 5.0)
    return {"spearman": spearmanr(labels, predictions)[0], "mse": mean_squared_error(labels, predictions)}

tokenize_func = lambda x: tokenizer(x["text"], padding="max_length", truncation=True, max_length=512)
tokenized_train = train_ds.map(tokenize_func, batched=True)
tokenized_dev = dev_ds.map(tokenize_func, batched=True)

args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=4,
    optim="paged_adamw_8bit",
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    compute_metrics=compute_metrics,
)

# ==========================================
# 5. הרצה
# ==========================================
print("🚀 Training Model A...")
trainer.train(resume_from_checkpoint=last_checkpoint)

print(f"💾 Saving final model to {output_dir}")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("🏆 Done!")

In [ ]:
# ==========================================
# 6. חיזוי על DEV בלבד (Validation Step)
# ==========================================
import json
from scipy.stats import spearmanr

# renamed path
output_dir = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V19"
model_id = "NousResearch/Meta-Llama-3-8B"
train_path = "/content/drive/MyDrive/Data mining/Text_mining/train.json"
dev_path = "/content/drive/MyDrive/Data mining/Text_mining/dev.json"

def get_dev_predictions(model, tokenizer, tokenized_dev_dataset):
    print("🔮 Running inference on DEV set...")
    # שימוש ב-Trainer הקיים לביצוע חיזוי
    predictions_output = trainer.predict(tokenized_dev_dataset)

    # חילוץ הציון וביצוע Clipping לטווח 1.0-5.0
    raw_preds = predictions_output.predictions.flatten()
    final_preds = np.clip(raw_preds, 1.0, 5.0)

    return final_preds

# הרצת החיזוי
dev_preds = get_dev_predictions(model, tokenizer, tokenized_dev)

# חישוב Spearman סופי למעקב
actual_labels = df_dev['label'].values
final_spearman = spearmanr(actual_labels, dev_preds)[0]

print(f"\n📊 Final DEV Spearman: {final_spearman:.4f}")

# שמירת התוצאות לקובץ JSON (מפתחות הם ה-index של ה-dev)
# זה ישמש אותנו אחר כך לאנסמבל מול מודל B
dev_results = {str(idx): float(pred) for idx, pred in zip(df_dev.index.tolist(), dev_preds)}

output_file = os.path.join(output_dir, "dev_preds_model_A.json")
with open(output_file, "w") as f:
    json.dump(dev_results, f, indent=4)

print(f"💾 DEV predictions saved to: {output_file}")

In [ ]:
import json
import os
import numpy as np
import pandas as pd
from transformers import Trainer, TrainingArguments

# ==========================================
# 6. חיזוי על DEV בפורמט JSONL (מתוקן)
# ==========================================

# הגדרת נתיב השמירה המדויק שביקשת
output_file_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V19/submission-v19.json"

# יצירת התיקייה במידה והיא לא קיימת
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

def run_dev_inference_and_save(model, tokenizer, tokenized_dev, df_dev, file_path):
    print(f"🔮 Starting inference on {len(df_dev)} samples...")

    # הגדרת arguments בסיסיים כדי למנוע שגיאות של Trainer חדש
    temp_args = TrainingArguments(output_dir="./temp", report_to="none")

    # הגדרת Trainer לחיזוי בלבד
    # הסרנו את tokenizer=tokenizer מה-init כדי למנוע את ה-TypeError
    trainer_inf = Trainer(
        model=model,
        args=temp_args
    )

    # ביצוע הניבוי
    predictions_output = trainer_inf.predict(tokenized_dev)

    # עיבוד התוצאות: שיטוח וקליפינג לטווח 1-5
    raw_preds = predictions_output.predictions.flatten()
    final_preds = np.clip(raw_preds, 1.0, 5.0)

    # בניית רשימת התוצאות בפורמט JSONL (שורה לכל אובייקט)
    dev_indices = df_dev.index.tolist()

    print(f"📝 Writing results to JSONL format...")

    with open(file_path, 'w', encoding='utf-8') as f:
        for i, idx in enumerate(dev_indices):
            entry = {
                "id": str(idx),
                "prediction": float(final_preds[i])
            }
            # כתיבת האובייקט כשורה אחת והוספת ירידת שורה
            f.write(json.dumps(entry) + '\n')

    print(f"✅ Success! File saved at: {file_path}")
    return final_preds

# הרצה (נשתמש ב-tokenized_dev וב-df_dev שכבר טעונים בזיכרון שלך)
dev_preds_final = run_dev_inference_and_save(model, tokenizer, tokenized_dev, df_dev, output_file_path)

# בדיקה קצרה
with open(output_file_path, 'r') as f:
    print(f"👀 Preview of first line: {f.readline().strip()}")

# Ensamble v19 & Model B v2

In [ ]:
import json
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

def load_and_fix_data(path):
    try:
        df = pd.read_json(path)
    except ValueError:
        df = pd.read_json(path, orient='index')
    if df.shape[1] > 100 and df.shape[0] < df.shape[1]:
        df = df.T
    if 'label' not in df.columns:
        df['label'] = df['average'] if 'average' in df.columns else df.get('score', 0.0)
    return df

# --- 1. טעינת נתונים ---
path_a = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V19/submission-v19.json"
path_b = "/content/drive/MyDrive/Data mining/Text_mining/ModelB/V2/preds_HYBRID_ENSEMBLE_DEV.jsonl"
dev_path = "/content/drive/MyDrive/Data mining/Text_mining/dev.json"

def load_jsonl_preds(path):
    preds = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            preds[str(data['id'])] = float(data['prediction'])
    return preds

preds_a = load_jsonl_preds(path_a)
preds_b = load_jsonl_preds(path_b)
df_dev = load_and_fix_data(dev_path)

# --- 2. יצירת DataFrame משותף מול הלייבלים האמיתיים ---
# נשתמש ב-df_dev שכבר טעון לך בזיכרון מהאימון של מודל A
eval_data = []
for idx, row in df_dev.iterrows():
    s_idx = str(idx)
    if s_idx in preds_a and s_idx in preds_b:
        eval_data.append({
            "id": s_idx,
            "actual": float(row['label']),
            "stdev": float(row.get('stdev', 1.0)),
            "pred_a": preds_a[s_idx],
            "pred_b": preds_b[s_idx]
        })

df_eval = pd.DataFrame(eval_data)

# --- 3. חיפוש המשקל האופטימלי (Grid Search) ---
best_acc = 0
best_weight = 0
results = []

print(f"🔍 Searching for optimal weights (ModelA * w + ModelB * (1-w))...")

for w in np.linspace(0, 1, 11): # בודק 0.0, 0.1, ..., 1.0
    ensemble_preds = df_eval['pred_a'] * w + df_eval['pred_b'] * (1 - w)
    ensemble_preds = np.clip(ensemble_preds, 1.0, 5.0)

    # חישוב Accuracy (לפי הקריטריון של התחרות)
    errors = abs(df_eval['actual'] - ensemble_preds)
    thresholds = df_eval['stdev'].apply(lambda x: max(x, 1.0))
    acc = (errors <= thresholds).mean()

    # חישוב Spearman
    spearman = spearmanr(df_eval['actual'], ensemble_preds)[0]

    results.append({"weight_a": w, "accuracy": acc, "spearman": spearman})

    if acc > best_acc:
        best_acc = acc
        best_weight = w

# --- 4. הצגת תוצאות ---
res_df = pd.DataFrame(results)
print("\n📊 Ensemble Results on DEV:")
print(res_df.to_string(index=False))

print(f"\n🏆 Best Weight for Model A: {best_weight}")
print(f"🏆 Best Ensemble Accuracy: {best_acc:.2%}")

🔍 Searching for optimal weights (ModelA * w + ModelB * (1-w))...

📊 Ensemble Results on DEV:
 weight_a  accuracy  spearman
      0.0  0.794218  0.666289
      0.1  0.802721  0.682569
      0.2  0.804422  0.696894
      0.3  0.809524  0.709244
      0.4  0.806122  0.717170
      0.5  0.811224  0.720532
      0.6  0.818027  0.721987
      0.7  0.814626  0.720124
      0.8  0.807823  0.714442
      0.9  0.804422  0.705695
      1.0  0.795918  0.694315

🏆 Best Weight for Model A: 0.6000000000000001
🏆 Best Ensemble Accuracy: 81.80%


In [ ]:
# 2. שדרוג pip והתקנת הספריות
!pip install --upgrade pip
!pip install -q -U bitsandbytes accelerate peft transformers datasets scipy scikit-learn

print("✅ Installation Done. Now please RESTART SESSION (Runtime -> Restart Session).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
✅ Installation Done. Now please RESTART SESSION (Runtime -> Restart Session).


In [ ]:
!pip install -U --no-cache-dir transformers datasets accelerate peft bitsandbytes

In [ ]:
import os
import torch
import pandas as pd
import json
import numpy as np
from datasets import Dataset, concatenate_datasets
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import PeftModel, PeftConfig

# ==========================================
# 1. הגדרת נתיבים (Paths Configuration)
# ==========================================
base_path = "/content/drive/MyDrive/Data mining/Text_mining"

# נתיב למודל הקיים (ממנו ממשיכים - איפה ששמור הצ'קפוינט של ה-0.69)
original_model_v19 = os.path.join(base_path, "ModelA/V19")

# נתיב פלט לאימון המאוחד (הגרסה הסופית)
final_output_dir = os.path.join(base_path, "ModelA/V19_Final_Submission")

# נתיבי דאטה
train_path = os.path.join(base_path, "train.json")
dev_path = os.path.join(base_path, "dev.json")
test_path = os.path.join(base_path, "test.json")

# קובץ הגשה סופי
submission_output_path = os.path.join(final_output_dir, "submission-v19-final.json")

os.makedirs(final_output_dir, exist_ok=True)

# ==========================================
# 2. פונקציות טעינה ועיבוד (עצמאיות)
# ==========================================
def load_and_fix_data(path):
    try:
        df = pd.read_json(path)
    except ValueError:
        df = pd.read_json(path, orient='index')
    if df.shape[1] > 100 and df.shape[0] < df.shape[1]:
        df = df.T
    if 'label' not in df.columns:
        df['label'] = df['average'] if 'average' in df.columns else df.get('score', 0.0)
    return df

def format_data(row):
    score = row.get('label', 0.0)
    precontext = str(row.get('precontext', '')).strip()
    sentence = str(row.get('sentence', '')).strip()
    ending = row.get('ending', '')
    homonym = row.get('homonym', '')
    definition = row.get('judged_meaning', '')

    has_ending = pd.notna(ending) and str(ending).strip() != ""
    if has_ending:
        instruction = "Task: Evaluate story consistency.\nScale: 1.0 to 5.0."
        context_text = f"Context: {precontext}\nSentence: {sentence}\nEnding: {ending}"
    else:
        instruction = "Task: Evaluate sentence fit.\nScale: 1.0 to 5.0."
        context_text = f"Context: {precontext}\nTarget Sentence: {sentence}"

    text = f"### Instruction:\n{instruction}\n\n### Target: {homonym} ({definition})\n\n### Narrative:\n{context_text}\n\n### Score (1-5):"
    return {"text": text, "label": float(score)}

# ==========================================
# 3. טעינת מודל וטוקנייזר (טעינה נקייה מהדיסק)
# ==========================================
print("🔄 Loading Model and Tokenizer from V19 Checkpoint...")
# טעינת ה-Base Model ב-4bit (כמו ב-V16/19 המקורי)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# מציאת הצ'קפוינט האחרון בתיקייה המקורית
checkpoints = [os.path.join(original_model_v19, d) for d in os.listdir(original_model_v19) if d.startswith("checkpoint")]
last_ckpt = sorted(checkpoints, key=lambda x: int(x.split('-')[-1]))[-1] if checkpoints else original_model_v19
print(f"📍 Resuming from weights at: {last_ckpt}")

tokenizer = AutoTokenizer.from_pretrained("NousResearch/Meta-Llama-3-8B")
tokenizer.pad_token = tokenizer.eos_token

# טעינת המודל עם ה-Peft (LoRA) שכבר אומן
model = AutoModelForSequenceClassification.from_pretrained(
    "NousResearch/Meta-Llama-3-8B",
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
    problem_type="regression"
)
model = PeftModel.from_pretrained(model, last_ckpt, is_trainable=True)
model.config.pad_token_id = tokenizer.pad_token_id

# ==========================================
# 4. הכנת הנתונים המאוחדים
# ==========================================
print("📊 Preparing Integrated Dataset (Train + DEV)...")
def tokenize_func(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

df_train = load_and_fix_data(train_path)
df_dev = load_and_fix_data(dev_path)

ds_train = Dataset.from_list([format_data(row) for _, row in df_train.iterrows()]).map(tokenize_func, batched=True)
ds_dev = Dataset.from_list([format_data(row) for _, row in df_dev.iterrows()]).map(tokenize_func, batched=True)

final_combined_dataset = concatenate_datasets([ds_train, ds_dev])

# ==========================================
# 5. הגדרות אימון סופיות
# ==========================================
# ==========================================
# 5. הגדרות אימון סופיות (תיקון לשגיאת BFloat16)
# ==========================================
# ==========================================
# 5. הגדרות אימון סופיות (תיקון ל-AMP ו-BFloat16)
# ==========================================

# ==========================================
# 5. הגדרות אימון סופיות (הגרסה היציבה ביותר)
# ==========================================

# 1. נחזיר את המודל למצב עבודה רגיל (מבטל את ה-.half() אם הורץ)
model.to(torch.float32)

final_args = TrainingArguments(
    output_dir=final_output_dir,
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    optim="paged_adamw_8bit",
    fp16=True,                        # ה-Trainer ינהל את ה-Scaling בעצמו
    bf16=False,
    gradient_checkpointing=True,
    # הגדרה קריטית למניעת שגיאות סקיילר ב-T4:
    fp16_full_eval=False,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=10,
    report_to="none"
)

# 2. אתחול ה-Trainer
final_trainer = Trainer(
    model=model,
    args=final_args,
    train_dataset=final_combined_dataset
)

# ==========================================
# 5. ריצת האימון (Stable FP16)
# ==========================================
print("🚀 Starting final combined training push...")
final_trainer.train()

# ==========================================
# 6. שמירה סופית ומלאה
# ==========================================
print(f"💾 Saving final model and tokenizer to {final_output_dir}")

# שמירת המודל (Weights + Config)
final_trainer.save_model(final_output_dir)

# שמירת הטוקנייזר - קריטי לשחזור עתידי של ה-Special Tokens
tokenizer.save_pretrained(final_output_dir)

print("🏆 Model A is fully saved and ready for inference!")

# ==========================================
# 6. חיזוי על ה-TEST בפורמט JSONL (שורה אחת)
# ==========================================
print(f"🔮 Inference on TEST data...")
df_test = load_and_fix_data(test_path)
test_ds = Dataset.from_list([format_data(row) for _, row in df_test.iterrows()])
tokenized_test = test_ds.map(tokenize_func, batched=True)

test_output = final_trainer.predict(tokenized_test)
test_preds = np.clip(test_output.predictions.flatten(), 1.0, 5.0)

with open(submission_output_path, 'w', encoding='utf-8') as f:
    for idx, pred in zip(df_test.index.tolist(), test_preds):
        f.write(json.dumps({"id": str(idx), "prediction": float(pred)}) + '\n')

print(f"🏆 Ready! Final submission saved at: {submission_output_path}")

🔄 Loading Model and Tokenizer from V19 Checkpoint...
📍 Resuming from weights at: /content/drive/MyDrive/Data mining/Text_mining/ModelA/V19/checkpoint-572


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

📊 Preparing Integrated Dataset (Train + DEV)...


Map:   0%|          | 0/2280 [00:00<?, ? examples/s]

Map:   0%|          | 0/588 [00:00<?, ? examples/s]

🚀 Starting final combined training push...


Step,Training Loss
10,10.485562
20,4.859515
30,4.938838
40,5.227423
50,4.370577
60,4.041700
70,7.140736
80,4.821528
90,5.119250
100,5.483079


continue from checkpint

In [ ]:
import os
import torch
import pandas as pd
import json
import numpy as np
from datasets import Dataset, concatenate_datasets
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import PeftModel

# ==========================================
# 1. הגדרת נתיבים (Paths Configuration)
# ==========================================
base_path = "/content/drive/MyDrive/Data mining/Text_mining"

# נתיב המקור (המודל הראשוני)
original_model_v19 = os.path.join(base_path, "ModelA/V19")

# נתיב היעד (האימון המאוחד שבו נמצא checkpoint-200)
final_output_dir = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V19_Final_Submission"

# נתיבי דאטה
train_path = os.path.join(base_path, "train.json")
dev_path = os.path.join(base_path, "dev.json")
test_path = os.path.join(base_path, "test.json")

submission_output_path = os.path.join(final_output_dir, "submission-v19-final.json")

os.makedirs(final_output_dir, exist_ok=True)

# ==========================================
# 2. פונקציות טעינה ועיבוד
# ==========================================
def load_and_fix_data(path):
    try:
        df = pd.read_json(path)
    except ValueError:
        df = pd.read_json(path, orient='index')
    if df.shape[1] > 100 and df.shape[0] < df.shape[1]:
        df = df.T
    if 'label' not in df.columns:
        df['label'] = df['average'] if 'average' in df.columns else df.get('score', 0.0)
    return df

def format_data(row):
    score = row.get('label', 0.0)
    precontext = str(row.get('precontext', '')).strip()
    sentence = str(row.get('sentence', '')).strip()
    ending = row.get('ending', '')
    homonym = row.get('homonym', '')
    definition = row.get('judged_meaning', '')

    has_ending = pd.notna(ending) and str(ending).strip() != ""
    if has_ending:
        instruction = "Task: Evaluate story consistency.\nScale: 1.0 to 5.0."
        context_text = f"Context: {precontext}\nSentence: {sentence}\nEnding: {ending}"
    else:
        instruction = "Task: Evaluate sentence fit.\nScale: 1.0 to 5.0."
        context_text = f"Context: {precontext}\nTarget Sentence: {sentence}"

    text = f"### Instruction:\n{instruction}\n\n### Target: {homonym} ({definition})\n\n### Narrative:\n{context_text}\n\n### Score (1-5):"
    return {"text": text, "label": float(score)}

# ==========================================
# 3. טעינת מודל וטוקנייזר
# ==========================================
print("🔄 Loading Base Model and Tokenizer...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained("NousResearch/Meta-Llama-3-8B")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    "NousResearch/Meta-Llama-3-8B",
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
    problem_type="regression"
)

# מציאת הצ'קפוינט האחרון ביותר מבין שתי התיקיות
all_ckpts = []
for d in [final_output_dir, original_model_v19]:
    if os.path.exists(d):
        all_ckpts.extend([os.path.join(d, c) for c in os.listdir(d) if c.startswith("checkpoint-")])

if all_ckpts:
    last_ckpt = sorted(all_ckpts, key=lambda x: int(x.split('-')[-1]))[-1]
    print(f"📍 Found latest checkpoint: {last_ckpt}")
else:
    last_ckpt = original_model_v19
    print(f"⚠️ No checkpoints found. Starting from original V19: {last_ckpt}")

model = PeftModel.from_pretrained(model, last_ckpt, is_trainable=True)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(torch.float32) # וידוא יציבות עבור ה-Scaler ב-T4

# ==========================================
# 4. הכנת הנתונים
# ==========================================
print("📊 Preparing Integrated Dataset (Train + DEV)...")
def tokenize_func(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

df_train = load_and_fix_data(train_path)
df_dev = load_and_fix_data(dev_path)

ds_train = Dataset.from_list([format_data(row) for _, row in df_train.iterrows()]).map(tokenize_func, batched=True)
ds_dev = Dataset.from_list([format_data(row) for _, row in df_dev.iterrows()]).map(tokenize_func, batched=True)
final_combined_dataset = concatenate_datasets([ds_train, ds_dev])

# ==========================================
# 5. הגדרות אימון - גרסת "עקיפת שגיאת אופטימייזר"
# ==========================================

final_args = TrainingArguments(
    output_dir=final_output_dir,
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,               # נסתפק באופק אחד כדי לסיים בטוח
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    save_strategy="steps",
    save_steps=20,                    # שמירה כל 20 צעדים
    save_total_limit=2,
    logging_steps=5,
    report_to="none"
)

final_trainer = Trainer(
    model=model,
    args=final_args,
    train_dataset=final_combined_dataset
)

print("🚀 Starting training (Weights loaded, Optimizer reset to fix ValueError)...")

# במקום resume_from_checkpoint=last_ckpt שגורם לשגיאה, פשוט נריץ train רגיל.
# המודל כבר מחזיק את המשקלים של last_ckpt בגלל שורת ה-PeftModel.from_pretrained למעלה.
final_trainer.train()

# שמירה סופית
final_trainer.save_model(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

# ==========================================
# 6. חיזוי על ה-TEST
# ==========================================
print(f"🔮 Inference on TEST data...")
df_test = load_and_fix_data(test_path)
test_ds = Dataset.from_list([format_data(row) for _, row in df_test.iterrows()])
tokenized_test = test_ds.map(tokenize_func, batched=True)

test_output = final_trainer.predict(tokenized_test)
test_preds = np.clip(test_output.predictions.flatten(), 1.0, 5.0)

with open(submission_output_path, 'w', encoding='utf-8') as f:
    for idx, pred in zip(df_test.index.tolist(), test_preds):
        f.write(json.dumps({"id": str(idx), "prediction": float(pred)}) + '\n')

print(f"🏆 Ready! Submission saved at: {submission_output_path}")

🔄 Loading Base Model and Tokenizer...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

LlamaForSequenceClassification LOAD REPORT from: NousResearch/Meta-Llama-3-8B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


📍 Found latest checkpoint: /content/drive/MyDrive/Data mining/Text_mining/ModelA/V19/checkpoint-572
📊 Preparing Integrated Dataset (Train + DEV)...


Map:   0%|          | 0/2280 [00:00<?, ? examples/s]

Map:   0%|          | 0/588 [00:00<?, ? examples/s]

🚀 Starting training (Weights loaded, Optimizer reset to fix ValueError)...


Step,Training Loss
5,4.053593
10,16.917529
15,6.259320
20,3.434056
25,4.128045
30,5.520966
35,4.176886
40,5.621339
45,3.308044
50,6.147548


🔮 Inference on TEST data...


ValueError: could not convert string to float: '(???)'

inference for test

In [ ]:
import json
import numpy as np
from datasets import Dataset

# 1. טעינת נתוני הטסט
test_path = "/content/drive/MyDrive/Data mining/Text_mining/test.json"
print(f"🔮 Loading TEST data from {test_path}...")
df_test = load_and_fix_data(test_path)

# 2. עיבוד הנתונים - כאן התיקון הקריטי
test_list = []
for _, row in df_test.iterrows():
    # יוצרים עותק של השורה כדי לא לדרוס את המקור
    temp_row = row.copy()

    # מכריחים את ה-label להיות 0.0 עבור ה-format_data
    # זה מונע את השגיאה של (???) כי הפונקציה תקבל מספר תקין
    temp_row['label'] = 0.0

    formatted = format_data(temp_row)
    test_list.append(formatted)

test_ds = Dataset.from_list(test_list)
tokenized_test = test_ds.map(tokenize_func, batched=True)

# 3. הרצת החיזוי
print("🏃 Running inference on TEST set...")
predictions_output = final_trainer.predict(tokenized_test)

# 4. עיבוד התוצאות (Clipping לטווח 1-5)
test_preds = np.clip(predictions_output.predictions.flatten(), 1.0, 5.0)

# 5. שמירה לפורמט JSONL
submission_output_path = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V19_Final_Submission/submission-v19-final.json"

print(f"📝 Saving predictions to: {submission_output_path}")
test_indices = df_test.index.tolist()

with open(submission_output_path, 'w', encoding='utf-8') as f:
    for i, idx in enumerate(test_indices):
        entry = {
            "id": str(idx),
            "prediction": float(test_preds[i])
        }
        f.write(json.dumps(entry) + '\n')

print("🏆 Done! Test inference is complete.")

🔮 Loading TEST data from /content/drive/MyDrive/Data mining/Text_mining/test.json...


Map:   0%|          | 0/930 [00:00<?, ? examples/s]

🏃 Running inference on TEST set...


📝 Saving predictions to: /content/drive/MyDrive/Data mining/Text_mining/ModelA/V19_Final_Submission/submission-v19-final.json
🏆 Done! Test inference is complete.


ensammble with Model B V2

In [ ]:
import json
import numpy as np
import os

# ==========================================
# 1. הגדרת נתיבים לקבצי ה-TEST (עדכני במידת הצורך)
# ==========================================
# קובץ ה-TEST שנוצר מהמודל שאת מאמנת עכשיו (Model A)
path_test_a = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V19_Final_Submission/submission-v19-final.json"

# קובץ ה-TEST של מודל B (DeBERTa) מהשותפה
path_test_b = "/content/drive/MyDrive/Data mining/Text_mining/ModelB/V2/submission_HYBRID_FINAL_TEST.jsonl"

# נתיב לקובץ ההגשה הסופי
final_submission_path = "/content/drive/MyDrive/Data mining/Text_mining/FINAL_SUBMISSION_V19_COMBINED.json"

# ==========================================
# 2. פונקציית טעינה
# ==========================================
def load_results(path):
    data = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            # שמירת המזהה והחיזוי
            data[str(obj['id'])] = float(obj['prediction'])
    return data

# ==========================================
# 3. ביצוע המיזוג לפי המשקולות (A: 0.6, B: 0.4)
# ==========================================
print("🚀 Loading predictions...")
preds_a = load_results(path_test_a)
preds_b = load_results(path_test_b)

print(f"📊 Merging with weights: ModelA=0.6, ModelB=0.4")
final_output = []

# עוברים על כל המזהים הקיימים (מוודאים שהם קיימים בשני המודלים)
for idx in preds_a.keys():
    if idx in preds_b:
        # חישוב הממוצע המשוקלל [0.6*A + 0.4*B]
        combined_val = (preds_a[idx] * 0.6) + (preds_b[idx] * 0.4)

        # Clipping לטווח הרלוונטי (1-5)
        final_val = np.clip(combined_val, 1.0, 5.0)

        final_output.append({
            "id": idx,
            "prediction": float(final_val)
        })

# ==========================================
# 4. שמירה בפורמט JSONL להגשה
# ==========================================
with open(final_submission_path, 'w', encoding='utf-8') as f:
    for entry in final_output:
        f.write(json.dumps(entry) + '\n')

print(f"🏆 Done! Final submission file created at: {final_submission_path}")
print(f"✅ Total samples merged: {len(final_output)}")

🚀 Loading predictions...
📊 Merging with weights: ModelA=0.6, ModelB=0.4
🏆 Done! Final submission file created at: /content/drive/MyDrive/Data mining/Text_mining/FINAL_SUBMISSION_V19_COMBINED.json
✅ Total samples merged: 930


In [ ]:
import pandas as pd
import json
import numpy as np

# ==========================================
# 1. טעינת נתונים (השתמשי בקבצי ה-DEV שבו יש Ground Truth)
# ==========================================
def load_jsonl_to_dict(path):
    data = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            data[str(obj['id'])] = float(obj['prediction'])
    return data

# נתיבים - עדכני לנתיבים שלך
path_dev_a = "/content/drive/MyDrive/Data mining/Text_mining/ModelA/V19/submission-v19.json"
path_dev_b = "/content/drive/MyDrive/Data mining/Text_mining/ModelB/V2/preds_HYBRID_ENSEMBLE_DEV.jsonl"
path_actual = "/content/drive/MyDrive/Data mining/Text_mining/dev.json" # הקובץ המקורי עם ה-label

# טעינה
preds_a = load_jsonl_to_dict(path_dev_a)
preds_b = load_jsonl_to_dict(path_dev_b)
df_actual = pd.read_json(path_actual)
if df_actual.shape[1] > 100: df_actual = df_actual.T # תיקון מבנה אם צריך

# ==========================================
# 2. יצירת טבלת השוואה
# ==========================================
analysis_list = []

for idx, row in df_actual.iterrows():
    s_idx = str(idx)
    if s_idx in preds_a and s_idx in preds_b:
        actual = float(row['label'] if 'label' in row else row['average'])
        p_a = preds_a[s_idx]
        p_b = preds_b[s_idx]
        ensemble = (p_a * 0.6) + (p_b * 0.4)

        analysis_list.append({
            "id": s_idx,
            "text": row.get('sentence', '')[:100] + "...", # קיצור הטקסט לתצוגה
            "actual": actual,
            "pred_a": p_a,
            "pred_b": p_b,
            "ensemble": ensemble,
            "error_a": abs(p_a - actual),
            "error_b": abs(p_b - actual),
            "error_ensemble": abs(ensemble - actual),
            "disagreement": abs(p_a - p_b)
        })

df_analysis = pd.DataFrame(analysis_list)

# ==========================================
# 3. שליפת תובנות לדו"ח
# ==========================================

print("🔍 --- ניתוח דוגמאות לדוח הסופי --- \n")

# א. המקרה שבו האנסמבל הציל את המצב (מודל אחד טעה, השני צדק, והממוצע קרוב לאמת)
ensemble_wins = df_analysis[(df_analysis['error_ensemble'] < df_analysis['error_a']) &
                            (df_analysis['error_ensemble'] < df_analysis['error_b'])].head(3)
print("✅ דוגמאות בהן האנסמבל שיפר את התוצאה של שני המודלים:")
print(ensemble_wins[['id', 'actual', 'pred_a', 'pred_b', 'ensemble']].to_string())

print("\n" + "="*50 + "\n")

# ב. המקרה שבו המודלים הכי לא הסכימו (Disagreement)
conflicts = df_analysis.sort_values(by='disagreement', ascending=False).head(3)
print("⚔️ דוגמאות של אי-הסכמה קיצונית (אחד אומר 'עקבי' והשני 'לא עקבי'):")
print(conflicts[['id', 'actual', 'pred_a', 'pred_b', 'disagreement']].to_string())

print("\n" + "="*50 + "\n")

# ג. המקרים הכי קשים (שניהם טעו משמעותית)
hard_cases = df_analysis.sort_values(by='error_ensemble', ascending=False).head(3)
print("❌ הדוגמאות הכי קשות (שניהם פספסו את האמת):")
print(hard_cases[['id', 'actual', 'ensemble', 'error_ensemble']].to_string())

🔍 --- ניתוח דוגמאות לדוח הסופי --- 

✅ דוגמאות בהן האנסמבל שיפר את התוצאה של שני המודלים:
  id  actual    pred_a    pred_b  ensemble
3  3     4.2  4.007812  4.423828  4.174219
4  4     3.0  2.718750  3.266602  2.937891
9  9     3.8  3.341797  4.512695  3.810156


⚔️ דוגמאות של אי-הסכמה קיצונית (אחד אומר 'עקבי' והשני 'לא עקבי'):
      id  actual    pred_a    pred_b  disagreement
573  573     4.6  2.148438  4.540039      2.391602
186  186     2.2  2.617188  4.644531      2.027344
135  135     4.2  2.841797  4.866211      2.024414


❌ הדוגמאות הכי קשות (שניהם פספסו את האמת):
      id  actual  ensemble  error_ensemble
572  572     1.2  4.140234        2.940234
534  534     1.4  4.249609        2.849609
290  290     1.0  3.749219        2.749219


ניתוח תוצאות

In [ ]:
import json

# Load the files
with open('FINAL_SUBMISSION_V19_COMBINED (1).json', 'r', encoding='utf-8') as f:
    preds = json.load(f)
with open('test_with_labels.json', 'r', encoding='utf-8') as f:
    truths = json.load(f)

# Make ground truth easily searchable by ID
truth_dict = {item['id']: item for item in truths}

errors = []

# Compare predictions to ground truth
for pred in preds:
    item_id = pred['id']
    # Adjust the keys 'score' and 'label' if they are named differently in your specific JSONs
    predicted_score = pred.get('score', 0)

    if item_id in truth_dict:
        actual_score = truth_dict[item_id].get('label', 0)
        diff = predicted_score - actual_score
        abs_diff = abs(diff)

        errors.append({
            'id': item_id,
            'predicted': predicted_score,
            'actual': actual_score,
            'diff': diff,
            'abs_diff': abs_diff,
            # Fetch narrative text to understand the context of the error
            'text': truth_dict[item_id].get('text', '')
        })

# Sort by largest absolute error
errors.sort(key=lambda x: x['abs_diff'], reverse=True)

# Print top 5 errors for qualitative analysis
print("Top 5 Largest Errors (for Qualitative Case Studies):\n")
for i, err in enumerate(errors[:5]):
    print(f"Error #{i+1} | ID: {err['id']}")
    print(f"Predicted: {err['predicted']:.2f} | Actual: {err['actual']:.2f} | Diff: {err['diff']:.2f}")
    print(f"Narrative: {err['text'][:250]}...\n")
    print("-" * 50)